# 01 — Next-word prediction with an encoder–decoder Transformer

This notebook builds a small Transformer from random weights using simple PyTorch code.
You can read the attention, encoder, decoder, loss and training loop here; no pretrained neural model is used.

**Run cells in order.** The training cell runs five epochs by default. Use the CUDA Jupyter kernel locally, or select a GPU runtime in Colab.
The separate second notebook will use the saved sentence pairs and trained model for next-concept comparison.

With byte-pair encoding (BPE), the actual prediction unit is a **token**, which may be a word, a word part or punctuation.
Here “next-word prediction” means autoregressive next-token prediction.

| Input to the encoder | Input to the decoder | What the decoder learns to predict |
| --- | --- | --- |
| Previous one or two sentences | `BOS, next sentence except its last token` | `Next sentence, EOS` |

Both notebooks use the same prose sentence pairs from the existing topic-held-out document splits.
The full documentation, including code and equations, stays in `data/`. Code blocks, tables and headings are excluded from this **sentence-continuation experiment**.
An optional configuration also trains this notebook on full-text windows.


## 1. What we are building

~~~text
Previous sentences → token embeddings + positions → ENCODER
                                                    │ memory
                                                    ▼
BOS + earlier target tokens → embeddings + positions → DECODER → vocabulary logits
                                                       │
                         masked self-attention → cross-attention → feed-forward
~~~

An encoder can attend to all tokens in the supplied context. Its input never includes the target sentence.
The decoder sees only earlier target tokens because its self-attention mask hides future positions.
Cross-attention lets decoder queries read encoder keys and values.

For target tokens \(y_1,\ldots,y_T\) and context \(x\), the loss is
\[
-\frac{1}{T}\sum_t \log p(y_t \mid x, y_{<t}).
\]
This is conditional continuation likelihood; its perplexity is specific to this context/target construction.

Architecture reference: [Attention Is All You Need](https://arxiv.org/abs/1706.03762).
We use its encoder–decoder structure, sinusoidal positions, residual connections, layer normalization and ReLU feed-forward layers, at a much smaller scale.


## 2. Imports

The first cell installs missing runtime packages. All model code below uses ordinary tensors and small classes.


In [ ]:
import importlib.util
import os
import subprocess
import sys

requirements = {
    "torch": "torch>=2.4,<3",
    "tokenizers": "tokenizers>=0.21,<1",
    "matplotlib": "matplotlib>=3.8,<4",
}
missing = [package for module, package in requirements.items()
           if importlib.util.find_spec(module) is None]
if missing:
    if os.environ.get("SKLEARNLM_NO_INSTALL") == "1":
        raise RuntimeError(f"Missing dependencies: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

import hashlib
import json
import math
from pathlib import Path
import random
import re
import time
from urllib.request import Request, urlopen
import zipfile

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)


## 3. Configuration

`DATA_MODE = "sentences"` is the matched experiment for comparison with next-concept prediction.
`"all_text"` instead learns continuations of complete document text, including code; that is a separate experiment.
Use a fresh output directory when changing settings.

Training is enabled when you run the training cell. Final test evaluation is a separate flag: use validation to choose settings first.
The small CPU smoke configuration is for automated code checks, not a quality result.


In [ ]:
SMOKE_TEST = os.environ.get("SKLEARNLM_SMOKE") == "1"
SMOKE_DEVICE = os.environ.get("SKLEARNLM_SMOKE_DEVICE", "cpu")
RUN_TRAINING = True
RUN_TEST_EVALUATION = SMOKE_TEST  # Set True after choosing the model using validation.
DATA_MODE = "sentences"          # Or "all_text"; keep "sentences" for notebook 02.
SEED = 42

CFG = {
    "vocab_size": 8000,
    "source_length": 192,
    "target_length": 128,
    "context_sentences": 2,
    "d_model": 128,
    "n_heads": 4,
    "encoder_layers": 2,
    "decoder_layers": 2,
    "dropout": 0.1,
    "batch_size": 16,
    "epochs": 5,
    "learning_rate": 3e-4,
    "weight_decay": 0.01,
    "patience": 3,
    "max_steps": None,
}
if SMOKE_TEST and SMOKE_DEVICE == "cpu":
    CFG.update(vocab_size=512, source_length=64, target_length=48, d_model=32,
               encoder_layers=1, decoder_layers=1, dropout=0.0, batch_size=2,
               epochs=1, max_steps=2)

if SMOKE_TEST:
    CFG.update(epochs=1, max_steps=2)

DEVICE = torch.device(SMOKE_DEVICE if SMOKE_TEST else "cuda")
if DEVICE.type == "cuda" and not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU required. Select the SKlearnLM CUDA kernel and install the CUDA requirements, "
        "or enable a GPU runtime in Colab. This notebook will not silently train on CPU."
    )
assert DEVICE.type in ("cpu", "cuda")
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if SMOKE_TEST:
    torch.set_num_threads(2)

DATA_REVISION = "bb503b2b994518c4640e4d966392b03cb35b6cbe"
DATA_URL = f"https://raw.githubusercontent.com/vishal88736/SKlearnLM/{DATA_REVISION}/data/"
MANIFEST_SHA = "6956c22a78c59c712b53570d8e0e6c346744d1287746fdd6dbf6cd4272d399e7"
PAD, BOS, EOS = "<|pad|>", "<|start|>", "<|endoftext|>"
print("Device:", DEVICE, "| data mode:", DATA_MODE)
print(json.dumps(CFG, indent=2))


## GPU training and mixed precision

Normal training requires a CUDA GPU and stops with a clear message if one is unavailable.
Select the project's CUDA Jupyter kernel, or choose a GPU runtime in Colab.

Automatic mixed precision uses bfloat16 when the GPU supports it, otherwise float16 with gradient scaling.
Model weights stay in float32. Forward/loss computation uses autocast; backward runs outside that context.
Gradients are unscaled before clipping. Checkpoints retain the scaler state.

The CPU path is reserved for explicit automated smoke tests. A CUDA smoke test uses the normal model dimensions
and performs two actual optimizer updates. Device, precision and peak allocated GPU memory are recorded.
See [PyTorch AMP examples](https://docs.pytorch.org/docs/2.6/notes/amp_examples.html).


In [ ]:
USE_AMP = DEVICE.type == "cuda"
precision_choice = os.environ.get("SKLEARNLM_PRECISION", "auto")
assert precision_choice in ("auto", "bf16", "fp16", "fp32")
if precision_choice == "bf16" and USE_AMP and not torch.cuda.is_bf16_supported():
    raise RuntimeError("This GPU does not support bfloat16; use fp16 or auto.")
AMP_DTYPE = (torch.bfloat16 if USE_AMP and
             (precision_choice == "bf16" or
              (precision_choice == "auto" and torch.cuda.is_bf16_supported()))
             else torch.float16)
USE_AMP = USE_AMP and precision_choice != "fp32"

def amp_context():
    return torch.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP)

RUNTIME = {
    "device": DEVICE.type,
    "torch_version": str(torch.__version__),
    "cuda_runtime": torch.version.cuda,
    "mixed_precision": USE_AMP,
    "training_precision": str(AMP_DTYPE) if USE_AMP else "torch.float32",
}
if DEVICE.type == "cuda":
    RUNTIME["gpu_name"] = torch.cuda.get_device_name(DEVICE)
    RUNTIME["total_vram_bytes"] = torch.cuda.get_device_properties(DEVICE).total_memory
    with amp_context():
        probe = torch.ones(32, 32, device=DEVICE) @ torch.ones(32, 32, device=DEVICE)
    assert torch.isfinite(probe).all()
    if USE_AMP:
        assert probe.dtype == AMP_DTYPE
    torch.cuda.synchronize()
    del probe
print(json.dumps(RUNTIME, indent=2))


## 4. Load and verify the documentation

Local repository data is used when available. In a standalone Colab session, the cell downloads the same pinned dataset.
Every data file is checked against the recorded byte count and SHA-256. The manifest itself has a pinned hash.

The document splits were made before tokenization. Whole mapped topic families remain in one split.
The corpus audit found 28 generic prose blocks shared across splits; group separation is not proof of zero semantic overlap.


In [ ]:
def sha256(payload):
    return hashlib.sha256(payload).hexdigest()

override = os.environ.get("SKLEARNLM_DATA_DIR")
candidates = [Path.cwd() / "data", Path.cwd().parent / "data"]
DATA_DIR = (Path(override) if override else
            next((p for p in candidates if (p / "manifest.json").is_file()),
                 Path.cwd() / "sklearnlm_data"))
DATA_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = Path(os.environ.get("SKLEARNLM_RUN_DIR",
                             str(DATA_DIR.parent / "runs" / "next_word")))
RUN_DIR.mkdir(parents=True, exist_ok=True)

def fetch_missing(name):
    path = DATA_DIR / name
    if not path.exists():
        request = Request(DATA_URL + name, headers={"User-Agent": "SKlearnLM"})
        with urlopen(request, timeout=60) as response:
            payload = response.read()
        path.write_bytes(payload)
    return path.read_bytes()

manifest_bytes = fetch_missing("manifest.json")
assert sha256(manifest_bytes) == MANIFEST_SHA, "Unexpected dataset manifest."
manifest = json.loads(manifest_bytes)
texts = {}
for name in ("train.txt", "validation.txt", "test.txt", "documents.jsonl"):
    payload = fetch_missing(name)
    expected = manifest["files"][name]
    assert len(payload) == expected["bytes"] and sha256(payload) == expected["sha256"], name
    if name.endswith(".txt"):
        texts[name[:-4]] = payload.decode("utf-8")
    else:
        documents = [json.loads(line) for line in payload.decode("utf-8").splitlines()]

for split in ("train", "validation", "test"):
    count = sum(doc["split"] == split for doc in documents)
    assert count == manifest["statistics"]["splits"][split]["documents"]
    print(split, ":", count, "documents")
(RUN_DIR / "data_manifest.json").write_bytes(manifest_bytes)
print("Output directory:", RUN_DIR.resolve())


## 5. Train BPE on the training text only

Byte-level BPE starts with byte symbols and learns frequent merges. The full byte alphabet preserves Unicode, indentation and punctuation.
PAD fills short sequences, BOS starts the decoder, and EOS ends a target continuation.

The tokenizer is fit on `train.txt` only. Validation/test text never changes its vocabulary.
A saved tokenizer must match its training-data fingerprint and settings before reuse.


In [ ]:
bpe_started = time.perf_counter()
tokenizer_settings = {
    "trained_on": "train.txt",
    "training_file_sha256": manifest["files"]["train.txt"]["sha256"],
    "vocab_size_requested": CFG["vocab_size"],
    "special_tokens": [PAD, BOS, EOS],
    "minimum_frequency": 2,
}
tokenizer_path = RUN_DIR / "tokenizer.json"
settings_path = RUN_DIR / "tokenizer_metadata.json"
if tokenizer_path.exists():
    assert settings_path.exists(), "Tokenizer metadata is missing."
    assert json.loads(settings_path.read_text()) == tokenizer_settings, "Choose a fresh RUN_DIR."
    tokenizer = Tokenizer.from_file(str(tokenizer_path))
else:
    assert not (RUN_DIR / "best.pt").exists(), "Checkpoint exists without its tokenizer."
    tokenizer = Tokenizer(models.BPE())
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=CFG["vocab_size"], min_frequency=2, special_tokens=[PAD, BOS, EOS],
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=not SMOKE_TEST,
    )
    train_docs = [text for text in texts["train"].split(EOS) if text.strip()]
    tokenizer.train_from_iterator(train_docs, trainer=trainer, length=len(train_docs))
    tokenizer.save(str(tokenizer_path))
    settings_path.write_text(json.dumps(tokenizer_settings, indent=2) + "\n")

PAD_ID, BOS_ID, EOS_ID = [tokenizer.token_to_id(token) for token in (PAD, BOS, EOS)]
CFG["vocab_size"] = tokenizer.get_vocab_size()
TOKENIZER_SHA = sha256(tokenizer_path.read_bytes())
sample = "def square(x):\n    return x ** 2\n# café, Δ\n"
assert tokenizer.decode(tokenizer.encode(sample).ids) == sample
assert Tokenizer.from_file(str(tokenizer_path)).encode(sample).ids == tokenizer.encode(sample).ids
encoded = tokenizer.encode("Linear regression predicts a continuous value.")
print("BPE pieces:", encoded.tokens)
print("Token IDs: ", encoded.ids)
print("Vocabulary size:", CFG["vocab_size"])
for split, text in texts.items():
    print(split, "full-document token count:", len(tokenizer.encode(text).ids))

BPE_SECONDS = time.perf_counter() - bpe_started


## 6. Make context → next-sentence pairs

The same pairs will feed both objectives. Notebook 01 predicts each target token; notebook 02 predicts the target sentence embedding.

The small rule-based segmenter below starts new sections at headings and code fences. It uses sentence-ending punctuation as a heuristic.
It keeps prose units of 5–60 whitespace-separated words and does not connect sentences across excluded material.
Abbreviations and technical formatting can still confuse it; inspect examples before making research claims.

Pairs never cross document or split boundaries. Examples longer than the configured BPE limits are counted and excluded, rather than silently truncated.


In [ ]:
def sentence_sections(text):
    sections, current, paragraph = [], [], []
    fence = None

    def end_section():
        if current:
            sections.append(current.copy())
            current.clear()

    def end_paragraph():
        if not paragraph:
            return
        prose = " ".join(paragraph).strip()
        paragraph.clear()
        if prose.startswith(("[Figure:", "$$", "|", "..", ":")):
            end_section()
            return
        sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z("])', prose)
        for sentence in sentences:
            if 5 <= len(sentence.split()) <= 60 and len(sentence) < 800:
                current.append(sentence)
            else:
                end_section()

    for line in text.splitlines():
        line = line.strip()
        marker = re.match(r"^(\x60{3,}|~{3,})", line)
        if marker:
            end_paragraph()
            end_section()
            symbol = marker.group(1)
            if fence is None:
                fence = (symbol[0], len(symbol))
            elif symbol[0] == fence[0] and len(symbol) >= fence[1]:
                fence = None
            continue
        if fence:
            continue
        if line.startswith("#"):
            end_paragraph()
            end_section()
        elif not line:
            end_paragraph()
        else:
            bullet = re.match(r"^(?:[-*+] |\d+[.)] )", line)
            if bullet:
                end_paragraph()
                line = line[bullet.end():]
            paragraph.append(line)
    end_paragraph()
    end_section()
    return sections


In [ ]:
pairs_started = time.perf_counter()
pairs = {split: [] for split in ("train", "validation", "test")}
rejected = {split: 0 for split in pairs}
for doc in documents:
    for section_index, sentences in enumerate(sentence_sections(doc["text"])):
        for i in range(1, len(sentences)):
            context = sentences[max(0, i - CFG["context_sentences"]):i]
            source, target = " ".join(context), sentences[i]
            source_ids = tokenizer.encode(source).ids
            target_ids = tokenizer.encode(target).ids + [EOS_ID]
            split = doc["split"]
            if len(source_ids) > CFG["source_length"] or len(target_ids) > CFG["target_length"]:
                rejected[split] += 1
                continue
            pair_id = sha256(f"{doc['id']}:{section_index}:{i}".encode())[:24]
            pairs[split].append({
                "id": pair_id, "document_id": doc["id"], "split": split,
                "context": context, "source": source, "target": target,
                "source_ids": source_ids, "target_ids": target_ids,
            })

if SMOKE_TEST:
    for split in pairs:
        pairs[split] = pairs[split][:256 if split == "train" else 32]
assert all(pairs.values()), "No usable pairs; increase the sequence lengths."
pair_payload = "".join(json.dumps(row, ensure_ascii=False) + "\n"
                       for rows in pairs.values() for row in rows).encode("utf-8")
PAIRS_SHA = sha256(pair_payload)
(RUN_DIR / "sentence_pairs.jsonl").write_bytes(pair_payload)
pair_info = {"sha256": PAIRS_SHA, "smoke_test": SMOKE_TEST,
             "counts": {s: len(rows) for s, rows in pairs.items()},
             "length_rejections": rejected, "context_sentences": CFG["context_sentences"]}
(RUN_DIR / "pair_manifest.json").write_text(json.dumps(pair_info, indent=2) + "\n")
print(json.dumps(pair_info, indent=2))
print("\nCONTEXT:", pairs["train"][0]["source"])
print("TARGET: ", pairs["train"][0]["target"])

PAIR_SECONDS = time.perf_counter() - pairs_started


## 7. Teacher forcing and padding

For target `A B C EOS`, the decoder input is `BOS A B C`. Its labels are `A B C EOS`.
That one-position shift is teacher forcing. During generation we feed back the model's own predictions.

We pad only within each batch. Padding keys are masked out of attention, and padding labels do not contribute to the loss.
The optional full-text mode below uses document-local prefix/continuation windows; it never feeds a target token to the encoder.


In [ ]:
def full_text_rows(split):
    rows = []
    width = CFG["source_length"] + CFG["target_length"]
    for doc in documents:
        if doc["split"] != split:
            continue
        ids = tokenizer.encode(doc["text"]).ids + [EOS_ID]
        for start in range(0, len(ids), width):
            window = ids[start:start + width]
            if len(window) < 4:
                continue
            cut = max(1, len(window) - CFG["target_length"])
            rows.append({"source_ids": window[:cut],
                         "target_ids": window[cut:cut + CFG["target_length"]]})
    return rows

assert DATA_MODE in ("sentences", "all_text")
training_rows = pairs if DATA_MODE == "sentences" else {
    split: full_text_rows(split) for split in pairs
}

class ContinuationDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        target = row["target_ids"]
        source = torch.tensor(row["source_ids"], dtype=torch.long)
        decoder_input = torch.tensor([BOS_ID] + target[:-1], dtype=torch.long)
        labels = torch.tensor(target, dtype=torch.long)
        return source, decoder_input, labels

def collate(batch):
    source, decoder_input, labels = zip(*batch)
    return tuple(pad_sequence(tensors, batch_first=True, padding_value=PAD_ID)
                 for tensors in (source, decoder_input, labels))

loaders = {}
for split, rows in training_rows.items():
    loaders[split] = DataLoader(
        ContinuationDataset(rows), batch_size=CFG["batch_size"],
        shuffle=(split == "train"), collate_fn=collate, num_workers=0, pin_memory=(DEVICE.type == "cuda"),
        generator=torch.Generator().manual_seed(SEED),
    )
    print(split, "training/evaluation examples:", len(rows))

source, decoder_input, labels = next(iter(loaders["train"]))
assert torch.equal(decoder_input[:, 1:][labels[:, 1:] != PAD_ID],
                   labels[:, :-1][labels[:, 1:] != PAD_ID])
print("Encoder IDs:", source[0].tolist())
print("Decoder IDs:", decoder_input[0].tolist())
print("Labels:     ", labels[0].tolist())
print("Batch shapes:", source.shape, decoder_input.shape, labels.shape)


## 8. Token embeddings and positions

An embedding looks up a learned vector for each token ID. Attention alone does not encode word order.
We add fixed sine/cosine position vectors, following the original Transformer.

For hidden dimension \(d\):
\[
PE(p,2i)=\sin(p/10000^{2i/d}),\qquad
PE(p,2i+1)=\cos(p/10000^{2i/d}).
\]
The embedding vectors are scaled by \(\sqrt{d}\) before adding positions.


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_length, dropout):
        super().__init__()
        assert d_model % 2 == 0
        position = torch.arange(max_length).float().unsqueeze(1)
        frequency = torch.exp(torch.arange(0, d_model, 2).float()
                              * (-math.log(10000.0) / d_model))
        values = torch.zeros(max_length, d_model)
        values[:, 0::2] = torch.sin(position * frequency)
        values[:, 1::2] = torch.cos(position * frequency)
        self.register_buffer("values", values.unsqueeze(0))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(x + self.values[:, :x.size(1)])


In [ ]:
positions = PositionalEncoding(CFG["d_model"], CFG["source_length"], 0.0)
fig, ax = plt.subplots(figsize=(8, 3))
image = ax.imshow(positions.values[0, :40, :32].numpy(), aspect="auto", cmap="coolwarm")
ax.set(xlabel="Hidden dimension", ylabel="Position", title="Sinusoidal position vectors")
fig.colorbar(image, ax=ax)
fig.tight_layout()
fig.savefig(RUN_DIR / "positions.png")
plt.show()


## 9. Scaled dot-product attention, written explicitly

Each attention head makes queries \(Q\), keys \(K\) and values \(V\).
\[
A=\mathrm{softmax}\left(QK^\top/\sqrt{d_k}+\mathrm{mask}\right),\qquad Z=AV.
\]

The mask sets forbidden scores to negative infinity before softmax, so those probabilities become zero.
We calculate each operation below rather than calling a prebuilt Transformer block.

Shapes: input `[batch, length, d_model]` → heads `[batch, heads, length, head_dim]` →
attention weights `[batch, heads, query_length, key_length]`.


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.query = nn.Linear(d_model, d_model)
        self.key = nn.Linear(d_model, d_model)
        self.value = nn.Linear(d_model, d_model)
        self.output = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        batch, length, width = x.shape
        return x.reshape(batch, length, self.n_heads, self.head_dim).transpose(1, 2)

    def forward(self, query, key, value, mask=None):
        q = self.split_heads(self.query(query))
        k = self.split_heads(self.key(key))
        v = self.split_heads(self.value(value))
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        attended = self.dropout(weights) @ v
        batch, heads, length, head_dim = attended.shape
        attended = attended.transpose(1, 2).contiguous().reshape(batch, length, heads * head_dim)
        return self.output(attended), weights


## 10. The three masks

Here `True` means **allowed to attend**.

1. Encoder mask: source padding cannot be used as a key.
2. Decoder mask: target padding and future target tokens cannot be used as keys.
3. Cross-attention mask: the decoder cannot attend to source padding.

The encoder can see its entire supplied context, because every context token precedes the target sentence.
A bidirectional encoder receiving the target sentence would leak the answer; our dataset never does that.


In [ ]:
def make_masks(source, decoder_input):
    source_mask = (source != PAD_ID)[:, None, None, :]
    length = decoder_input.size(1)
    causal = torch.tril(torch.ones(length, length, dtype=torch.bool, device=source.device))
    decoder_mask = (decoder_input != PAD_ID)[:, None, None, :] & causal[None, None, :, :]
    return source_mask, decoder_mask

toy = torch.randn(1, 6, CFG["d_model"])
toy_attention = MultiHeadAttention(CFG["d_model"], CFG["n_heads"], 0.0)
allowed = torch.tril(torch.ones(6, 6, dtype=torch.bool))[None, None]
_, weights = toy_attention(toy, toy, toy, allowed)
assert torch.allclose(weights.sum(-1), torch.ones_like(weights.sum(-1)))
assert weights[0, :, torch.triu(torch.ones(6, 6, dtype=torch.bool), diagonal=1)].eq(0).all()

q = toy_attention.split_heads(toy_attention.query(toy))
k = toy_attention.split_heads(toy_attention.key(toy))
v = toy_attention.split_heads(toy_attention.value(toy))
manual = weights @ v
reference = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed, dropout_p=0.0)
assert torch.allclose(manual, reference, atol=1e-6)

fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(weights[0, 0].detach().numpy(), vmin=0, vmax=1, cmap="Blues")
ax.set(xlabel="Key position", ylabel="Query position", title="One untrained head: future weights are zero")
fig.tight_layout()
fig.savefig(RUN_DIR / "causal_attention.png")
plt.show()


## 11. Feed-forward network

Attention mixes information between positions. The feed-forward network transforms each position separately:
`Linear(d, 4d) → ReLU → Dropout → Linear(4d, d)`.

Layer normalization and residual additions will surround these sublayers in each block.


In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, dropout):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(4 * d_model, d_model),
        )

    def forward(self, x):
        return self.layers(x)


## 12. One encoder block

The encoder uses unmasked self-attention over the valid context tokens:
`attention → add residual + normalize → feed-forward → add residual + normalize`.

We use post-normalization, as in the original paper. A small stack of these blocks produces the encoder memory.


In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.feed_forward = FeedForward(d_model, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, source_mask):
        attended, _ = self.attention(x, x, x, source_mask)
        x = self.norm1(x + self.dropout(attended))
        x = self.norm2(x + self.dropout(self.feed_forward(x)))
        return x


## 13. One decoder block

The decoder has **three** sublayers.

- Masked self-attention reads previous target tokens.
- Cross-attention uses decoder states as queries and encoder memory as keys/values.
- A feed-forward network transforms the result.

Each sublayer has its own residual addition and layer normalization.


In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.feed_forward = FeedForward(d_model, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, memory, source_mask, decoder_mask):
        attended, _ = self.self_attention(x, x, x, decoder_mask)
        x = self.norm1(x + self.dropout(attended))
        attended, _ = self.cross_attention(x, memory, memory, source_mask)
        x = self.norm2(x + self.dropout(attended))
        x = self.norm3(x + self.dropout(self.feed_forward(x)))
        return x


## 14. Assemble the encoder–decoder

The source and decoder share a token embedding table because they use the same language/vocabulary.
The output linear layer maps each decoder state to one logit per vocabulary token.
Cross-entropy handles the softmax during training; generation applies softmax explicitly when sampling.


In [ ]:
class WordTransformer(nn.Module):
    def __init__(self, config):
        super().__init__()
        width = config["d_model"]
        self.scale = math.sqrt(width)
        self.embedding = nn.Embedding(config["vocab_size"], width, padding_idx=PAD_ID)
        max_length = max(config["source_length"], config["target_length"]) + 1
        self.positions = PositionalEncoding(width, max_length, config["dropout"])
        self.encoder = nn.ModuleList([
            EncoderBlock(width, config["n_heads"], config["dropout"])
            for _ in range(config["encoder_layers"])
        ])
        self.decoder = nn.ModuleList([
            DecoderBlock(width, config["n_heads"], config["dropout"])
            for _ in range(config["decoder_layers"])
        ])
        self.output = nn.Linear(width, config["vocab_size"])

    def encode(self, source, source_mask):
        memory = self.positions(self.embedding(source) * self.scale)
        for block in self.encoder:
            memory = block(memory, source_mask)
        return memory

    def decode(self, decoder_input, memory, source_mask, decoder_mask):
        x = self.positions(self.embedding(decoder_input) * self.scale)
        for block in self.decoder:
            x = block(x, memory, source_mask, decoder_mask)
        return self.output(x)

    def forward(self, source, decoder_input):
        source_mask, decoder_mask = make_masks(source, decoder_input)
        memory = self.encode(source, source_mask)
        return self.decode(decoder_input, memory, source_mask, decoder_mask)

model = WordTransformer(CFG).to(DEVICE)
PARAMETERS = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nTrainable parameters: {PARAMETERS:,}")
for name in ("embedding", "encoder", "decoder", "output"):
    print(name, ":", sum(p.numel() for p in getattr(model, name).parameters()))


## 15. Check the model before training

These checks verify tensor shapes, finite gradients, decoder causality, source-padding invariance and dependence on encoder context.
The earlier attention check also compares our explicit calculation with PyTorch's scaled attention calculation.

Changing a later target token must not change an earlier output. Changing a context token should change the output.
None of these checks demonstrates learned language ability.


In [ ]:
source, decoder_input, labels = [x.to(DEVICE) for x in next(iter(loaders["train"]))]
model.eval()
with torch.no_grad():
    logits = model(source, decoder_input)
    assert logits.shape == (*decoder_input.shape, CFG["vocab_size"])
    probe = torch.tensor([[BOS_ID, 10, 11, 12]], device=DEVICE)
    changed = probe.clone()
    changed[:, 2:] = torch.tensor([13, 14], device=DEVICE)
    original_logits = model(source[:1], probe)
    assert torch.allclose(original_logits[:, :2], model(source[:1], changed)[:, :2], atol=1e-5)
    padded_source = F.pad(source[:1], (0, 1), value=PAD_ID)
    assert torch.allclose(original_logits, model(padded_source, probe), atol=1e-5)
    other_source = source[:1].clone()
    other_source[:, 0] = 15 if int(other_source[0, 0]) != 15 else 16
    assert not torch.allclose(original_logits, model(other_source, probe), atol=1e-6)

# Recompute with gradients: the diagnostic forward above was inference-only.
loss = F.cross_entropy(model(source, decoder_input).reshape(-1, CFG["vocab_size"]),
                       labels.reshape(-1), ignore_index=PAD_ID)
loss.backward()
assert all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)
model.zero_grad(set_to_none=True)
checks = {"shape": True, "causal_mask": True, "source_padding": True,
          "cross_attention_uses_context": True, "finite_gradients": True,
          "manual_attention_matches_pytorch": True}
if USE_AMP:
    with torch.no_grad(), amp_context():
        amp_logits = model(source, decoder_input)
    assert amp_logits.dtype == AMP_DTYPE and torch.isfinite(amp_logits).all()
    checks["mixed_precision_forward"] = True
(RUN_DIR / "model_checks.json").write_text(json.dumps(checks, indent=2) + "\n")
print(checks)
print("Initial loss on one batch:", float(loss.detach()))


## 16. Evaluate with token-weighted loss

We sum the loss over non-padding target tokens and divide by their count.
Averaging batch means would give short batches too much weight.

Perplexity is \(\exp(\mathrm{cross\ entropy})\). Token accuracy is measured with teacher forcing.
Both depend on BPE and the continuation task; neither directly measures factual correctness.


In [ ]:
@torch.no_grad()
@torch.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP)
def evaluate(model, loader, max_batches=None):
    was_training = model.training
    model.eval()
    total_loss, correct, tokens = 0.0, 0, 0
    for batch_index, batch in enumerate(loader):
        if max_batches is not None and batch_index >= max_batches:
            break
        source, decoder_input, labels = [x.to(DEVICE, non_blocking=True) for x in batch]
        logits = model(source, decoder_input)
        valid = labels != PAD_ID
        total_loss += F.cross_entropy(
            logits.reshape(-1, CFG["vocab_size"]), labels.reshape(-1),
            ignore_index=PAD_ID, reduction="sum",
        ).item()
        correct += ((logits.argmax(-1) == labels) & valid).sum().item()
        tokens += valid.sum().item()
    model.train(was_training)
    assert tokens > 0
    cross_entropy = total_loss / tokens
    return {"cross_entropy": cross_entropy,
            "perplexity": math.exp(cross_entropy) if cross_entropy < 80 else float("inf"),
            "token_accuracy": correct / tokens, "target_tokens": tokens,
            "scope": "two_batches_only" if max_batches == 2 else "full_split"}


## 17. Save enough information to reload the result

Checkpoints contain weights, optimizer state, architecture settings, data/tokenizer fingerprints and measured history.
A separate inference checkpoint will omit optimizer state.

The configuration guard prevents accidentally mixing runs with different settings.
Checkpoint loading below restores a selected model; exact interrupted-run continuation is not implemented.


In [ ]:
experiment = {"objective": "next_token", "architecture": "encoder_decoder",
              "data_mode": DATA_MODE, "config": CFG, "seed": SEED,
              "training_precision": RUNTIME["training_precision"],
              "smoke_test": SMOKE_TEST, "tokenizer_sha256": TOKENIZER_SHA,
              "pairs_sha256": PAIRS_SHA, "data_manifest_sha256": MANIFEST_SHA}
config_path = RUN_DIR / "experiment.json"
if config_path.exists():
    assert json.loads(config_path.read_text()) == experiment, "Use a fresh RUN_DIR."
else:
    config_path.write_text(json.dumps(experiment, indent=2) + "\n")

def save_checkpoint(path, optimizer, scaler, step, history):
    torch.save({
        "model_state": model.state_dict(), "optimizer_state": optimizer.state_dict(),
        "scaler_state": scaler.state_dict(), "runtime": RUNTIME,
        "experiment": experiment, "step": step, "history": history,
        "torch_version": str(torch.__version__),
    }, path)


## 18. Train next-token prediction

The loop is deliberately direct: forward → cross-entropy → backward → clip gradients → AdamW step.
Each epoch measures validation loss. Only validation selects `best.pt`; the test split is never used for selection.

The default uses a constant learning rate to keep the code easy to follow.
Reduce the learning rate or model size if validation worsens while training loss keeps falling.
For research comparisons, repeat runs with several seeds and report variability.


In [ ]:
def train():
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["learning_rate"],
                                 weight_decay=CFG["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and AMP_DTYPE == torch.float16)
    history, step, best_loss, stale_epochs = [], 0, float("inf"), 0
    skipped_updates = 0
    start_time = time.perf_counter()
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    for epoch in range(1, CFG["epochs"] + 1):
        model.train()
        total_loss, total_tokens = 0.0, 0
        for batch in loaders["train"]:
            source, decoder_input, labels = [x.to(DEVICE, non_blocking=True) for x in batch]
            optimizer.zero_grad(set_to_none=True)
            with amp_context():
                logits = model(source, decoder_input)
                loss = F.cross_entropy(logits.reshape(-1, CFG["vocab_size"]),
                                       labels.reshape(-1), ignore_index=PAD_ID)
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite training loss.")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0,
                                     error_if_nonfinite=not scaler.is_enabled())
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() < old_scale:
                skipped_updates += 1
                continue  # Overflow: GradScaler skipped this optimizer update.
            count = (labels != PAD_ID).sum().item()
            total_loss += loss.item() * count
            total_tokens += count
            step += 1
            if step % 100 == 0:
                print(f"epoch {epoch}, step {step}: batch loss {loss.item():.4f}")
            if CFG["max_steps"] is not None and step >= CFG["max_steps"]:
                break
        assert total_tokens > 0, "No finite optimizer updates; try bfloat16 or float32."
        validation = evaluate(model, loaders["validation"], 2 if SMOKE_TEST else None)
        row = {"epoch": epoch, "step": step, "train_loss": total_loss / total_tokens,
               "validation_loss": validation["cross_entropy"],
               "validation_perplexity": validation["perplexity"],
               "elapsed_seconds": time.perf_counter() - start_time,
               "amp_skipped_updates": skipped_updates}
        history.append(row)
        print(row)
        save_checkpoint(RUN_DIR / "last.pt", optimizer, scaler, step, history)
        if row["validation_loss"] < best_loss:
            best_loss, stale_epochs = row["validation_loss"], 0
            save_checkpoint(RUN_DIR / "best.pt", optimizer, scaler, step, history)
        else:
            stale_epochs += 1
        (RUN_DIR / "history.json").write_text(json.dumps(history, indent=2) + "\n")
        if stale_epochs >= CFG["patience"]:
            print("Early stopping: validation did not improve.")
            break
        if CFG["max_steps"] is not None and step >= CFG["max_steps"]:
            break
    return history

history = train() if RUN_TRAINING else []
if not RUN_TRAINING:
    print("Training skipped. Existing matching checkpoints may be evaluated below.")


## 19. Plot measured loss

These plots are created from this run's history. Smoke-run curves contain only two optimizer updates.


In [ ]:
if history:
    fig, ax = plt.subplots(figsize=(7, 4))
    epochs = [row["epoch"] for row in history]
    ax.plot(epochs, [row["train_loss"] for row in history], "o-", label="Training")
    ax.plot(epochs, [row["validation_loss"] for row in history], "o-", label="Validation")
    ax.set(xlabel="Epoch", ylabel="Cross-entropy per target token",
           title="Next-token encoder–decoder training")
    ax.legend()
    fig.tight_layout()
    fig.savefig(RUN_DIR / "loss_curve.png")
    plt.show()


## 20. Restore the best validation checkpoint

We check the saved experiment identity and verify that reloading produces the same logits.
Final validation uses every validation batch in a normal run.


In [ ]:
metrics = {"objective": "next_token", "architecture": "encoder_decoder",
           "smoke_test": SMOKE_TEST, "parameters": PARAMETERS, "runtime": RUNTIME,
           "measured_preprocessing_seconds": BPE_SECONDS + PAIR_SECONDS,
           "experiment": experiment, "full_training_performed": not SMOKE_TEST and bool(history)}
best_path = RUN_DIR / "best.pt"
if best_path.exists():
    checkpoint = torch.load(best_path, map_location=DEVICE, weights_only=True)
    assert checkpoint["experiment"] == experiment
    model.load_state_dict(checkpoint["model_state"])
    model.eval()
    restored = WordTransformer(CFG).to(DEVICE)
    restored.load_state_dict(checkpoint["model_state"])
    restored.eval()
    with torch.no_grad():
        assert torch.allclose(model(source, decoder_input),
                              restored(source, decoder_input), atol=1e-6)
    del restored
    metrics["validation"] = evaluate(model, loaders["validation"], 2 if SMOKE_TEST else None)
    metrics["optimizer_updates"] = history[-1]["step"] if history else checkpoint["step"]
    metrics["selected_checkpoint_step"] = checkpoint["step"]
    metrics["training_seconds"] = history[-1]["elapsed_seconds"] if history else None
    metrics["peak_cuda_bytes"] = (torch.cuda.max_memory_allocated() if DEVICE.type == "cuda" else None)
    print(json.dumps(metrics["validation"], indent=2))
else:
    print("No trained checkpoint: run the training cell before reporting metrics.")


## 21. Final held-out evaluation

Leave this flag off while tuning. Enable it once the experiment design and model are fixed.
A smoke evaluation checks only two batches and must not be reported as full test performance.


In [ ]:
if RUN_TEST_EVALUATION and best_path.exists():
    metrics["test"] = evaluate(model, loaders["test"], 2 if SMOKE_TEST else None)
    print(json.dumps(metrics["test"], indent=2))
else:
    print("Final test evaluation is disabled or no trained checkpoint is available.")
(RUN_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")


## 22. Predict a continuation without teacher forcing

Encode the context once. Start the decoder with BOS, predict a token, append it, and repeat until EOS or the target limit.
Greedy decoding chooses the largest logit. A positive temperature samples from a softened distribution.

The displayed example comes from validation. Short training runs can produce repetition or incorrect technical claims.


In [ ]:
@torch.no_grad()
@torch.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP)
def generate(model, source_text, max_new_tokens=None, temperature=0.0):
    model.eval()
    ids = tokenizer.encode(source_text).ids
    if not ids or len(ids) > CFG["source_length"]:
        raise ValueError("Supply a nonempty context within the configured source length.")
    source = torch.tensor([ids], device=DEVICE)
    source_mask = (source != PAD_ID)[:, None, None, :]
    memory = model.encode(source, source_mask)
    generated = torch.tensor([[BOS_ID]], device=DEVICE)
    limit = min(max_new_tokens or CFG["target_length"], CFG["target_length"])
    for _ in range(limit):
        _, decoder_mask = make_masks(source, generated)
        logits = model.decode(generated, memory, source_mask, decoder_mask)[:, -1]
        logits[:, [PAD_ID, BOS_ID]] = float("-inf")
        if temperature > 0:
            next_token = torch.multinomial(torch.softmax(logits / temperature, dim=-1), 1)
        else:
            next_token = logits.argmax(-1, keepdim=True)
        generated = torch.cat((generated, next_token), dim=1)
        if next_token.item() == EOS_ID:
            break
    return tokenizer.decode(generated[0, 1:].tolist(), skip_special_tokens=True)

if best_path.exists():
    example = pairs["validation"][0]
    continuation = generate(model, example["source"], max_new_tokens=12 if SMOKE_TEST else None)
    print("CONTEXT:  ", example["source"])
    print("REFERENCE:", example["target"])
    print("GENERATED:", continuation)
    (RUN_DIR / "generation.json").write_text(json.dumps({
        "pair_id": example["id"], "context": example["source"],
        "reference": example["target"], "prediction": continuation,
        "smoke_test": SMOKE_TEST,
    }, ensure_ascii=False, indent=2) + "\n")


## 23. Save the model and hand off to notebook 02

The ZIP contains inference weights, tokenizer, exact sentence pairs, configuration, fingerprints and measured logs.
It contains your trained weights, not a pretrained model.

Notebook 02 reads this run directory so both experiments use exactly the same accepted sentence pairs.
If moving between Colab sessions, save/download the ZIP, extract it into a directory, and set `SKLEARNLM_WORD_RUN_DIR` there in notebook 02.


In [ ]:
if best_path.exists():
    inference = {"model_state": model.state_dict(), "experiment": experiment,
                 "pad_id": PAD_ID, "bos_id": BOS_ID, "eos_id": EOS_ID}
    torch.save(inference, RUN_DIR / "inference_model.pt")
    names = ["inference_model.pt", "tokenizer.json", "tokenizer_metadata.json",
             "sentence_pairs.jsonl", "pair_manifest.json", "experiment.json",
             "data_manifest.json", "metrics.json", "history.json", "model_checks.json"]
    with zipfile.ZipFile(RUN_DIR / "next_word_inference.zip", "w", zipfile.ZIP_DEFLATED) as archive:
        for name in names:
            if (RUN_DIR / name).exists():
                archive.write(RUN_DIR / name, arcname=name)
    print("Saved:", RUN_DIR / "next_word_inference.zip")
    print("Next: open 02_Next_Concept_Prediction.ipynb with WORD_RUN_DIR =", RUN_DIR)


## 24. What to record for the comparison

Record the seed, pair fingerprint, tokenizer, parameter count, number of optimizer updates, wall time and best validation checkpoint.
For multiple trials, keep the pair/tokenizer settings fixed and vary only the model initialization/shuffle seed.

The concept notebook has a different loss and uses a pretrained SONAR encoder/decoder.
Compare their training costs and shared held-out continuation measures; token cross-entropy and concept MSE are different quantities.
No training curves or quality scores are prefilled in this notebook.

The supplied [reference notebook](https://drive.google.com/file/d/1qyydJT5PC0eK4B0pXlbJ3N1f3gPKA6i6/view) informed the learning sequence.
This implementation changes its GPT-only model to the requested encoder–decoder and uses newly written explanations.
